In [0]:
# COMMAND ----------
import os
# import torch
import gc

# --- Widgets (Job parameters) ---
dbutils.widgets.text("model_type", "ffn")
dbutils.widgets.text("seed", "13")
dbutils.widgets.text("lr", "0.001")
dbutils.widgets.text("max_len", "256")
dbutils.widgets.text("base_batch_size", "128")
dbutils.widgets.text("epochs", "5")
dbutils.widgets.text("dataset", "20news")

# Cuántas GPUs usar en ESTE run (DDP world_size)
dbutils.widgets.text("num_processes", "2")

# local_mode=True => "Started local training with X processes" (normal en single-node multiGPU)
dbutils.widgets.dropdown("local_mode", "true", ["true", "false"])

# MLflow
dbutils.widgets.text("experiment_path", "/Users/david.grana@boehringer-ingelheim.com/side_project")
# Dataset cache (evita re-descargas)
dbutils.widgets.text("data_home", "/tmp/sklearn_data")

# Version old
# dbutils.widgets.dropdown("enable_mlflow", "true", ["true", "false"])
# # 🔥 MUY IMPORTANTE para paralelizar runs en el mismo nodo:
# # - Para 2 runs paralelo x 2 GPUs: usa "0,1" y "2,3"
# # - Para 4 runs paralelo x 1 GPU: usa "0" "1" "2" "3"
# dbutils.widgets.text("gpu_ids", "")  # vacío => todas visibles (no recomendado si hay concurrencia)

# # --- Aplicar pinning de GPU (afecta a procesos hijos torchrun) ---
# gpu_ids = dbutils.widgets.get("gpu_ids").strip()
# if gpu_ids:
#     os.environ["CUDA_VISIBLE_DEVICES"] = gpu_ids

# # Evita oversubscription CPU cuando lanzas muchos procesos
# os.environ.setdefault("OMP_NUM_THREADS", "1")

# Widgets...
dbutils.widgets.dropdown("local_mode", "true", ["true", "false"])
dbutils.widgets.text("gpu_ids", "")

local_mode = dbutils.widgets.get("local_mode").lower() == "true"
gpu_ids = dbutils.widgets.get("gpu_ids").strip()
_base = int(gpu_ids.split(",")[0]) if gpu_ids else 0
os.environ["MASTER_PORT"] = str(29500 + _base)

if local_mode:
    # TorchDistributor local_mode controla CUDA_VISIBLE_DEVICES internamente
    if "CUDA_VISIBLE_DEVICES" in os.environ:
        print("[DEBUG] Removing inherited CUDA_VISIBLE_DEVICES for TorchDistributor local_mode:",
              os.environ["CUDA_VISIBLE_DEVICES"], flush=True)
        del os.environ["CUDA_VISIBLE_DEVICES"]
else:
    # Incluso en local_mode=False normalmente es Spark quien gestiona GPUs en executors.
    # Si quieres mantener gpu_ids, hazlo bajo tu responsabilidad:
    if gpu_ids:
        os.environ["CUDA_VISIBLE_DEVICES"] = gpu_ids

os.environ.setdefault("OMP_NUM_THREADS", "1")

print("local_mode =", local_mode, flush=True)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"), flush=True)
print("OMP_NUM_THREADS =", os.environ.get("OMP_NUM_THREADS"), flush=True)


# --- Asegura que torchrun hereda credenciales para MLflow en Databricks ---
# (esto evita errores de auth cuando mlflow.set_tracking_uri("databricks") se ejecuta dentro de procesos)
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    os.environ["DATABRICKS_HOST"] = ctx.apiUrl().get()
    os.environ["DATABRICKS_TOKEN"] = ctx.apiToken().get()
    os.environ["MLFLOW_ENABLE_DB_SDK"] = "true"
except Exception:
    # Si por algún motivo no hay contexto (raro en Jobs), se ignora
    pass

local_mode = True
CUDA_VISIBLE_DEVICES = None
OMP_NUM_THREADS = 1


In [0]:
import re
import sys
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

sys.path.append(f"/Workspace/Users/david.grana@boehringer-ingelheim.com/side_project/vavava")
os.chdir(f"/Workspace/Users/david.grana@boehringer-ingelheim.com/side_project/vavava")
# ============================================================
# Loader dataset custom CSV (columnas TEXT / TEMA)
# ============================================================
def load_csv_topics(cfg: dict):
    csv_path  = "df_ml_topic_31_03_24.csv"                 # e.g. "/dbfs/FileStore/df_ml_topic_31_03_24.csv"
    text_col  = "TEXT"
    label_col = "TEMA"

    # (opcional) filtrar por lista de temas
    temas_principales = ['lugo', 'galicia', 'a-marina', 'deporte-local-lugo', 'a-chaira', 'ribeira-sacra', 'espana', 'gente', 'sarria', 'mundo', 'cultura', 'sociedad', 'comarcas', 'deporte-general', 'economia']
    seed = int(cfg.get("seed", 42))

    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[text_col, label_col]).reset_index(drop=True)

    if temas_principales is not None:
        df = df[df[label_col].isin(temas_principales)].reset_index(drop=True)

    # etiquetas -> ints consecutivos 0..K-1
    if not np.issubdtype(df[label_col].dtype, np.number):
        unique_labels = sorted(df[label_col].unique().tolist())
        label2id = {lbl: i for i, lbl in enumerate(unique_labels)}
        df[label_col] = df[label_col].map(label2id)
        label_names = unique_labels
    else:
        df[label_col] = df[label_col].astype(int)
        classes_sorted = sorted(df[label_col].unique().tolist())
        if classes_sorted != list(range(len(classes_sorted))):
            remap = {c: i for i, c in enumerate(classes_sorted)}
            df[label_col] = df[label_col].map(remap)
            classes_sorted = sorted(df[label_col].unique().tolist())
        label_names = [f"class_{i}" for i in classes_sorted]

    num_labels = len(label_names)

    X = df[text_col].astype(str).tolist()
    y = df[label_col].tolist()

    # split 80/20 y luego 10% del 80 para val
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, random_state=seed, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.10, random_state=seed, stratify=y_temp
    )

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), num_labels, label_names


# ============================================================
# Wrapper general: decide dataset por cfg["dataset"]
# ============================================================
def load_dataset(cfg: dict):
    ds = cfg.get("dataset", "20news").lower()

    if ds == "20news":
        data_home = cfg.get("data_home", "/tmp/sklearn_20news")
        (X_tr, y_tr), (X_va, y_va), (X_te, y_te), num_labels = load_20news(data_home)
        label_names = [f"class_{i}" for i in range(num_labels)]

        # patrón “inglés” (incluye dígitos si quieres)
        token_pattern = cfg.get("token_pattern", r"[A-Za-z0-9]+")
        return (X_tr, y_tr), (X_va, y_va), (X_te, y_te), num_labels, label_names, token_pattern

    elif ds in ("csv_topics", "custom_csv", "csv"):
        (X_tr, y_tr), (X_va, y_va), (X_te, y_te), num_labels, label_names = load_csv_topics(cfg)

        # patrón “español” por defecto (puedes sobreescribirlo con cfg)
        token_pattern = cfg.get("token_pattern", r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]+(?:-[A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]+)*")
        return (X_tr, y_tr), (X_va, y_va), (X_te, y_te), num_labels, label_names, token_pattern

    else:
        raise ValueError(f"dataset no soportado: {ds}")

In [0]:
# COMMAND ----------
import re
import numpy as np
from collections import Counter

TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+")
PAD, CLS, UNK = 0, 1, 2

def tok(text: str):
    return [t.lower() for t in TOKEN_PATTERN.findall(text)]

def build_vocab_deterministic(texts, min_freq=2, max_vocab=20000):
    c = Counter()
    for t in texts:
        c.update(tok(t))
    items = [(w, n) for w, n in c.items() if n >= min_freq]
    items.sort(key=lambda x: (-x[1], x[0]))  # determinista
    itos = ["<pad>", "<cls>", "<unk>"] + [w for w, _ in items][: max_vocab - 3]
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

def numericalize(text, stoi, max_len: int):
    ids = [CLS] + [stoi.get(t, UNK) for t in tok(text)]
    ids = ids[:max_len]
    attn = [1] * len(ids)
    if len(ids) < max_len:
        pad = max_len - len(ids)
        ids += [PAD] * pad
        attn += [0] * pad
    return np.asarray(ids, np.int64), np.asarray(attn, np.int64)

def suggest_batch_size(base_bs: int, max_len: int) -> int:
    return max(4, int(base_bs * (256 / max_len)))

In [0]:
# COMMAND ----------
import torch
from torch.utils.data import Dataset

class TextDS(Dataset):
    def __init__(self, X, y, stoi, max_len):
        self.X, self.y, self.stoi, self.max_len = X, y, stoi, max_len
    def __len__(self): 
        return len(self.X)
    def __getitem__(self, i):
        ids, attn = numericalize(self.X[i], self.stoi, self.max_len)
        return {"input_ids": ids, "attention_mask": attn, "labels": int(self.y[i])}

def collate(batch):
    input_ids = torch.stack([torch.tensor(b["input_ids"], dtype=torch.long) for b in batch])
    attn      = torch.stack([torch.tensor(b["attention_mask"], dtype=torch.float32) for b in batch])
    labels    = torch.tensor([b["labels"] for b in batch], dtype=torch.long)
    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}

In [0]:
# # COMMAND ----------
# import torch.nn as nn
# import torch

# def build_model(model_type: str, vocab_size: int, num_labels: int, max_len: int):
#     class FFN(nn.Module):
#         def __init__(self):
#             super().__init__()
#             self.emb = nn.Embedding(vocab_size, 128, padding_idx=PAD)
#             self.mlp = nn.Sequential(
#                 nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.2),
#                 nn.Linear(256, num_labels)
#             )
#         def forward(self, input_ids, attention_mask=None, labels=None):
#             x = self.emb(input_ids)
#             if attention_mask is not None:
#                 m = attention_mask.unsqueeze(-1).to(x.dtype)
#                 x = (x * m).sum(1) / m.sum(1).clamp_min(1.0)
#             else:
#                 x = x.mean(1)
#             logits = self.mlp(x)
#             loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
#             return loss, logits

#     class LSTM(nn.Module):
#         def __init__(self):
#             super().__init__()
#             self.emb = nn.Embedding(vocab_size, 128, padding_idx=PAD)
#             self.lstm = nn.LSTM(128, 256, batch_first=True, bidirectional=True)
#             self.fc = nn.Linear(512, num_labels)
#             self.drop = nn.Dropout(0.3)
#         def forward(self, input_ids, attention_mask=None, labels=None):
#             x = self.emb(input_ids)
#             if attention_mask is None:
#                 lengths = torch.full((input_ids.size(0),), input_ids.size(1),
#                                      dtype=torch.long, device=input_ids.device).cpu()
#             else:
#                 lengths = attention_mask.sum(1).long().cpu()
#             packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
#             _, (h, _) = self.lstm(packed)
#             h = torch.cat([h[-2], h[-1]], dim=1)
#             logits = self.fc(self.drop(h))
#             loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
#             return loss, logits

#     class TRF(nn.Module):
#         def __init__(self):
#             super().__init__()
#             self.emb = nn.Embedding(vocab_size, 128, padding_idx=PAD)
#             self.pos = nn.Embedding(max_len, 128)  # <-- reactivado
#             enc = nn.TransformerEncoderLayer(
#                 d_model=128, nhead=2, dim_feedforward=256, dropout=0.2,
#                 batch_first=True
#             )
#             self.enc = nn.TransformerEncoder(enc, num_layers=2)
#             self.fc = nn.Linear(128, num_labels)

#         def forward(self, input_ids, attention_mask=None, labels=None):
#             B, L = input_ids.shape
#             pos = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
#             x = self.emb(input_ids) + self.pos(pos)

#             kpm = (attention_mask == 0) if attention_mask is not None else None
#             x = self.enc(x, src_key_padding_mask=kpm)

#             if attention_mask is not None:
#                 m = attention_mask.unsqueeze(-1).to(x.dtype)
#                 x = (x * m).sum(1) / m.sum(1).clamp_min(1.0)
#             else:
#                 x = x.mean(1)

#             logits = self.fc(x)
#             loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
#             return loss, logits

#     if model_type == "ffn":
#         return FFN()
#     if model_type == "lstm":
#         return LSTM()
#     if model_type == "transformer_scratch":
#         return TRF()
#     raise ValueError(f"model_type no soportado: {model_type}")

In [0]:
# COMMAND ----------
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Modelos de CLASIFICACIÓN alineados con el capítulo 7
# (NextToken_WikiText). Mismos bloques que en modelado de lenguaje:
#   - Transformer: atención causal manual + pre-norm + ln_f + PE absoluta aprendida
#   - LSTM: unidireccional de 2 capas
#   - FFN: bag-of-words (media de embeddings)
# Hiperparámetros idénticos al cap. 7: d_model=256, n_heads=8,
# n_layers=4, ffn_dim=1024, dropout=0.1, lstm_layers=2.
#
# Única diferencia intrínseca a la tarea de clasificación: en lugar de
# una cabeza de lenguaje por token, se usa una cabeza de clasificación
# sobre la representación del ÚLTIMO TOKEN VÁLIDO (padding a la derecha).
# Con atención causal + padding a la derecha, ese token solo ha atendido
# a tokens reales, por lo que no se necesita máscara de padding extra.
# ============================================================

ARCH = dict(
    d_model     = 256,
    n_heads     = 8,
    n_layers    = 4,
    ffn_dim     = 1024,
    dropout     = 0.1,
    lstm_layers = 2,
)

def _last_valid(x, attention_mask):
    # x: [B, L, D] -> [B, D] tomando el último token válido por secuencia
    lengths = attention_mask.long().sum(1).clamp_min(1)          # [B]
    idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, x.size(-1)) # [B,1,D]
    return x.gather(1, idx).squeeze(1)

def _masked_mean(x, attention_mask):
    m = attention_mask.unsqueeze(-1).to(x.dtype)
    return (x * m).sum(1) / m.sum(1).clamp_min(1.0)

class CausalSelfAttention(nn.Module):
    """Idéntica a VanillaCausalSelfAttention del cap. 7."""
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        att = att.masked_fill(causal, float("-inf"))
        w = torch.softmax(att, dim=-1)
        w = F.dropout(w, p=self.dropout, training=self.training)
        y = (w @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.out(y)

class TransformerBlock(nn.Module):
    """Idéntico a VanillaTransformerBlock del cap. 7 (pre-norm + GELU)."""
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.att = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )
    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

def build_model(model_type: str, vocab_size: int, num_labels: int, max_len: int):
    d_model     = ARCH["d_model"]
    n_heads     = ARCH["n_heads"]
    n_layers    = ARCH["n_layers"]
    ffn_dim     = ARCH["ffn_dim"]
    dropout     = ARCH["dropout"]
    lstm_layers = ARCH["lstm_layers"]

    class Head(nn.Module):
        # Cabeza de clasificación común a los 3 modelos (misma capacidad)
        def __init__(self, in_dim):
            super().__init__()
            self.drop = nn.Dropout(dropout)
            self.fc = nn.Linear(in_dim, num_labels)
        def forward(self, x):
            return self.fc(self.drop(x))

    class FFN(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
            self.head = Head(d_model)
        def forward(self, input_ids, attention_mask=None, labels=None):
            x = self.emb(input_ids)
            x = _masked_mean(x, attention_mask) if attention_mask is not None else x.mean(1)
            logits = self.head(x)
            loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
            return loss, logits

    class LSTM(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
            self.lstm = nn.LSTM(d_model, d_model, num_layers=lstm_layers,
                                dropout=dropout if lstm_layers > 1 else 0.0,
                                batch_first=True)
            self.head = Head(d_model)
        def forward(self, input_ids, attention_mask=None, labels=None):
            x = self.emb(input_ids)
            if attention_mask is None:
                lengths = torch.full((input_ids.size(0),), input_ids.size(1),
                                     dtype=torch.long, device=input_ids.device).cpu()
            else:
                lengths = attention_mask.long().sum(1).clamp_min(1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            out, _ = self.lstm(packed)
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=input_ids.size(1))
            h = _last_valid(out, attention_mask) if attention_mask is not None else out[:, -1, :]
            logits = self.head(h)
            loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
            return loss, logits

    class TRF(nn.Module):
        def __init__(self):
            super().__init__()
            self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
            self.pos_emb = nn.Embedding(max_len, d_model)   # PE absoluta aprendida (igual que cap.7)
            self.drop = nn.Dropout(dropout)
            self.blocks = nn.ModuleList([
                TransformerBlock(d_model, n_heads, ffn_dim, dropout) for _ in range(n_layers)
            ])
            self.ln_f = nn.LayerNorm(d_model)               # ln_f final (igual que cap.7)
            self.head = Head(d_model)
        def forward(self, input_ids, attention_mask=None, labels=None):
            B, T = input_ids.shape
            pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
            x = self.drop(self.tok_emb(input_ids) + self.pos_emb(pos))
            for blk in self.blocks:
                x = blk(x)
            x = self.ln_f(x)
            h = _last_valid(x, attention_mask) if attention_mask is not None else x[:, -1, :]
            logits = self.head(h)
            loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
            return loss, logits

    if model_type == "ffn":
        return FFN()
    if model_type == "lstm":
        return LSTM()
    if model_type == "transformer_scratch":
        return TRF()
    raise ValueError(f"model_type no soportado: {model_type}")


In [0]:
# COMMAND ----------
import torch.distributed as dist
import torch

def all_gather_1d(t: torch.Tensor, world_size: int, pad_value: int = -1):
    """all_gather para tensores 1D con padding en el MISMO device."""
    device = t.device
    sz = torch.tensor([t.numel()], device=device)
    sz_list = [torch.zeros_like(sz) for _ in range(world_size)]
    dist.all_gather(sz_list, sz)
    max_sz = int(torch.stack(sz_list).max().item())

    if t.numel() < max_sz:
        pad = torch.full((max_sz - t.numel(),), pad_value, dtype=t.dtype, device=device)
        t = torch.cat([t, pad], 0)

    out = [torch.empty_like(t) for _ in range(world_size)]
    dist.all_gather(out, t)
    return out

def ddp_init():
    import os
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl", init_method="env://")
    return local_rank, rank, world_size

def ddp_cleanup():
    try:
        dist.barrier()
    except Exception:
        pass
    try:
        dist.destroy_process_group()
    except Exception:
        pass

In [0]:
# COMMAND ----------
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

def load_20news(data_home: str):
    train_raw = fetch_20newsgroups(subset="train", remove=("headers","footers","quotes"), data_home=data_home)
    test_raw  = fetch_20newsgroups(subset="test",  remove=("headers","footers","quotes"), data_home=data_home)

    X_tr, X_va, y_tr, y_va = train_test_split(
        train_raw.data, train_raw.target, test_size=0.1, random_state=42, stratify=train_raw.target
    )
    X_te, y_te = test_raw.data, test_raw.target
    num_labels = len(train_raw.target_names)

    return (X_tr, y_tr), (X_va, y_va), (X_te, y_te), num_labels

In [0]:
# COMMAND ----------
from dataclasses import dataclass, field
from typing import Any, Dict, Callable, Optional
import mlflow

def flatten_dict(d: Dict[str, Any], parent: str = "", sep: str = ".") -> Dict[str, Any]:
    """Aplana dicts anidados para log_params."""
    out = {}
    for k, v in d.items():
        key = f"{parent}{sep}{k}" if parent else str(k)
        if isinstance(v, dict):
            out.update(flatten_dict(v, key, sep=sep))
        else:
            out[key] = v
    return out

@dataclass
class MLflowTracker:
    enabled: bool
    is_main: bool
    experiment_path: str
    run_name: str
    artifact_dir: str = "artifacts"
    tags: Dict[str, Any] = field(default_factory=dict)

    def start(self, params: Dict[str, Any]) -> None:
        if not (self.enabled and self.is_main):
            return
        try:
            mlflow.set_experiment(self.experiment_path)
            mlflow.start_run(run_name=self.run_name)
            for k, v in self.tags.items():
                mlflow.set_tag(str(k), str(v))
            self.log_params(params)
        except Exception as e:
            print(f"[rank0] MLflow disabled in start(): {e}", flush=True)
            self.enabled = False

    def log_params(self, params: Dict[str, Any]) -> None:
        if not (self.enabled and self.is_main):
            return
        flat = flatten_dict(params)
        # mlflow.log_params exige strings
        mlflow.log_params({k: str(v) for k, v in flat.items()})

    def log_metrics(self, metrics: Dict[str, float], step: Optional[int] = None) -> None:
        if not (self.enabled and self.is_main):
            return
        mlflow.log_metrics(metrics, step=step)

    def log_dict(self, d: Dict[str, Any], artifact_file: str) -> None:
        if not (self.enabled and self.is_main):
            return
        mlflow.log_dict(d, artifact_file)

    def log_text(self, text: str, artifact_file: str) -> None:
        if not (self.enabled and self.is_main):
            return
        mlflow.log_text(text, artifact_file)

    def log_figure(self, fig, artifact_file: str) -> None:
        """
        Guarda figura como artifact (recomendado para visualizaciones MLflow). [1](https://mlflow.org/docs/latest/ml/traditional-ml/tutorials/hyperparameter-tuning/notebooks/logging-plots-in-mlflow)[2](https://apxml.com/courses/data-versioning-experiment-tracking/chapter-3-tracking-experiments-mlflow/logging-artifacts-mlflow)
        """
        if not (self.enabled and self.is_main):
            return
        mlflow.log_figure(fig, artifact_file)

    def end(self) -> None:
        if self.enabled and self.is_main:
            try:
                mlflow.end_run()
            except Exception:
                pass

In [0]:
# COMMAND ----------
import os, tempfile
import matplotlib.pyplot as plt

def plot_curves(tracker, history, prefix=""):
    ep = history["epoch"]

    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # --- Loss plot ---
    ax[0].plot(ep, history["train_loss"], label="train_loss")
    if "val_loss" in history:
        ax[0].plot(ep, history["val_loss"], label="val_loss")
    ax[0].set_title("Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("Loss")
    ax[0].legend()

    # --- Metrics plot ---
    ax[1].plot(ep, history["val_acc"], label="val_accuracy")
    ax[1].plot(ep, history["val_f1"], label="val_macro_f1")
    if "val_recall" in history:
        ax[1].plot(ep, history["val_recall"], label="val_macro_recall")
    ax[1].set_title("Validation metrics")
    ax[1].set_xlabel("Epoch")
    ax[1].legend()

    fig.tight_layout()

    d = tempfile.mkdtemp()
    out = os.path.join(d, f"{prefix}curves.png")
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)

    if tracker.enabled:
        try:
            tracker.log_artifact(out)
        except Exception:
            import mlflow
            mlflow.log_artifact(out, artifact_path="plots")

import os, tempfile
import numpy as np
import matplotlib.pyplot as plt

def plot_confusion(tracker, cm, class_names=None, prefix=""):
    class_names = class_names or [str(i) for i in range(cm.shape[0])]

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    fig.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="Real",
        xlabel="Predicción",
        title="Confusion Matrix"
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # (opcional) anotar valores
    thresh = cm.max() / 2.0 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")

    fig.tight_layout()

    # guardar y loggear
    d = tempfile.mkdtemp()
    out = os.path.join(d, f"{prefix}confusion_matrix.png")
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)

    if tracker.enabled:
        # si tu tracker tiene log_artifact(s) úsalo; si no, usa mlflow.log_artifact
        try:
            tracker.log_artifact(out)
        except Exception:
            import mlflow
            mlflow.log_artifact(out, artifact_path="plots")

def plot_len_hist(tracker: MLflowTracker, lengths, prefix: str = ""):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(lengths, bins=60)
    ax.set_title("Token length distribution"); ax.set_xlabel("len"); ax.set_ylabel("count")
    tracker.log_figure(fig, f"{tracker.artifact_dir}/{prefix}length_hist.png")
    plt.close(fig)

# Registro central (añade aquí lo que quieras)
PLOTTERS = {
    "curves": plot_curves,
    "confusion": plot_confusion,
    "length_hist": plot_len_hist,
}

In [0]:
# COMMAND ----------
import os, tempfile, traceback
import numpy as np
import torch
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.metrics import f1_score, recall_score

def train_ddp_one_cfg(cfg: dict):
    dist_inited = False
    tracker = None

    try:
        local_rank, rank, world_size = ddp_init()
        dist_inited = True
        is_main = (rank == 0)

        # --- seeds ---
        seed = int(cfg.get("seed", 42))
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)

        # --- datos + vocab ---
        data_home = cfg.get("data_home", "/tmp/sklearn_20news")
        # --- datos (según cfg['dataset']) ---
        (X_tr, y_tr), (X_va, y_va), (X_te, y_te), num_labels, label_names, token_pattern = load_dataset(cfg)

        # IMPORTANTE: actualizar TOKEN_PATTERN global usado por tok()/build_vocab_deterministic
        global TOKEN_PATTERN
        TOKEN_PATTERN = re.compile(token_pattern)

        stoi, itos = build_vocab_deterministic(
            X_tr,
            min_freq=int(cfg.get("min_freq", 2)),
            max_vocab=int(cfg.get("max_vocab", 20000))
        )
        vocab_size = len(itos)

        max_len = int(cfg["max_len"])
        base_bs = int(cfg["base_batch_size"])
        bs = suggest_batch_size(base_bs, max_len)

        train_ds = TextDS(X_tr, y_tr, stoi, max_len)
        val_ds   = TextDS(X_va, y_va, stoi, max_len)
        test_ds  = TextDS(X_te, y_te, stoi, max_len)

        train_sampler = DistributedSampler(train_ds, shuffle=True, drop_last=False)
        val_sampler   = DistributedSampler(val_ds,   shuffle=False, drop_last=False)
        test_sampler  = DistributedSampler(test_ds,  shuffle=False, drop_last=False)

        train_loader = DataLoader(train_ds, batch_size=bs, sampler=train_sampler, num_workers=0, pin_memory=True, collate_fn=collate)
        val_loader   = DataLoader(val_ds,   batch_size=bs, sampler=val_sampler,   num_workers=0, pin_memory=True, collate_fn=collate)
        test_loader  = DataLoader(test_ds,  batch_size=bs, sampler=test_sampler,  num_workers=0, pin_memory=True, collate_fn=collate)

        # --- modelo ---
        model_type = cfg["model_type"]
        dataset = cfg["dataset"]
        base_model = build_model(model_type, vocab_size, num_labels, max_len).cuda(local_rank)
        model = DDP(base_model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)

        # --- optim/amp ---
        lr = float(cfg["lr"])
        epochs = int(cfg["epochs"])
        opt = torch.optim.AdamW(model.parameters(), lr=lr)
        use_amp = bool(cfg.get("use_amp", False))
        scaler = torch.amp.GradScaler('cuda') if use_amp else None

        # ---------------------------
        # MLflow: EXTENSIBLE
        # ---------------------------
        enable_mlflow = bool(cfg.get("enable_mlflow", True))
        run_name = cfg.get("run_name", f"{model_type}-data{dataset}-seed{seed}-lr{lr}-L{max_len}")

        tracker = MLflowTracker(
            enabled=enable_mlflow,
            is_main=is_main,
            experiment_path=cfg["experiment_path"],
            run_name=run_name,
            artifact_dir=cfg.get("artifact_dir", "artifacts"),
            tags=cfg.get("tags", {})
        )

        # params base + extras (sin tocar código cada vez)
        base_params = {
            "model_type": model_type, "seed": seed, "lr": lr,
            "epochs": epochs, "max_len": max_len,
            "vocab_size": vocab_size, "batch_size_per_gpu": bs,
            "world_size": world_size, "use_amp": use_amp,
            "data_home": data_home,
            "dataset": dataset,
            "token_pattern": token_pattern
        }
        base_params.update(cfg.get("extra_params", {}))
        tracker.start(base_params)

        # qué plotters ejecutar al final
        plotter_names = cfg.get("plotters", ["curves", "confusion", "length_hist"])
        # frecuencia de logging (para no pasarte de límites) [3](https://docs.databricks.com/aws/en/mlflow/tracking)
        log_every = int(cfg.get("log_every_epochs", 1))

        # --- train/val ---
        history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "val_recall": []} if is_main else None
        best_f1 = -1.0

        for ep in range(1, epochs + 1):
            train_sampler.set_epoch(ep)
            model.train()

            ep_loss = torch.tensor(0.0, device=local_rank)
            ep_n = torch.tensor(0, device=local_rank)

            for batch in train_loader:
                opt.zero_grad(set_to_none=True)
                ids  = batch["input_ids"].cuda(local_rank, non_blocking=True)
                attn = batch["attention_mask"].cuda(local_rank, non_blocking=True)
                y    = batch["labels"].cuda(local_rank, non_blocking=True)

                if use_amp:
                    with torch.amp.autocast('cuda', enabled=True):
                        loss, _ = model(ids, attn, y)
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update()
                else:
                    loss, _ = model(ids, attn, y)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()

                ep_loss += loss.detach() * ids.size(0)
                ep_n += ids.size(0)

            # reduce loss global
            dist.all_reduce(ep_loss, op=dist.ReduceOp.SUM)
            dist.all_reduce(ep_n, op=dist.ReduceOp.SUM)
            train_loss = (ep_loss / ep_n.clamp_min(1)).item()

            model.eval()
            preds_local, gts_local = [], []

            val_loss_sum = torch.tensor(0.0, device=local_rank)
            val_n = torch.tensor(0, device=local_rank)

            with torch.no_grad():
                for batch in val_loader:
                    ids  = batch["input_ids"].cuda(local_rank, non_blocking=True)
                    attn = batch["attention_mask"].cuda(local_rank, non_blocking=True)
                    y    = batch["labels"].cuda(local_rank, non_blocking=True)

                    if use_amp:
                        with torch.amp.autocast('cuda', enabled=True):
                            loss, logits = model(ids, attn, y)   # 👈 pasamos labels para loss
                    else:
                        loss, logits = model(ids, attn, y)

                    val_loss_sum += loss.detach() * ids.size(0)
                    val_n += ids.size(0)

                    preds_local.append(logits.argmax(-1))
                    gts_local.append(y)

            # reduce global val_loss
            dist.all_reduce(val_loss_sum, op=dist.ReduceOp.SUM)
            dist.all_reduce(val_n, op=dist.ReduceOp.SUM)
            val_loss = (val_loss_sum / val_n.clamp_min(1)).item()

            preds_local = torch.cat(preds_local) if preds_local else torch.empty(0, dtype=torch.long, device=local_rank)
            gts_local   = torch.cat(gts_local)   if gts_local   else torch.empty(0, dtype=torch.long, device=local_rank)

            preds_all = all_gather_1d(preds_local, world_size, -1)
            gts_all   = all_gather_1d(gts_local,   world_size, -1)

            if is_main:
                p = torch.cat(preds_all).cpu().numpy()
                g = torch.cat(gts_all).cpu().numpy()
                m = (g != -1)
                p, g = p[m], g[m]

                val_acc = accuracy_score(g, p) if p.size else 0.0
                val_f1  = f1_score(g, p, average="macro") if p.size else 0.0
                val_rec = recall_score(g, p, average="macro") if p.size else 0.0

                history["epoch"].append(ep)
                history["train_loss"].append(train_loss)
                history["val_loss"].append(val_loss)
                history["val_acc"].append(val_acc)
                history["val_f1"].append(val_f1)
                history["val_recall"].append(val_rec)

                if (ep % log_every) == 0:
                    tracker.log_metrics(
                        {
                            "train_loss": train_loss,
                            "val_loss": val_loss,
                            "val_accuracy": val_acc,
                            "val_macro_f1": val_f1,
                            "val_macro_recall": val_rec,
                        },
                        step=ep
                    )

                # guardar mejor modelo
                if val_f1 > best_f1:
                    best_f1 = val_f1
                    if tracker.enabled:
                        d = tempfile.mkdtemp()
                        torch.save(model.module.state_dict(), os.path.join(d, "best.pt"))
                        mlflow.log_artifacts(d, artifact_path="best_model")

                print(f"[rank0] ep={ep} train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                    f"val_acc={val_acc:.4f} val_f1={val_f1:.4f} val_rec={val_rec:.4f}", flush=True)

        # ---------------------------
        # TEST + GRÁFICAS + SUMMARY
        # ---------------------------
        do_test = bool(cfg.get("do_test", True))
        if is_main:
            # curvas
            if tracker.enabled and ("curves" in plotter_names) and history is not None:
                PLOTTERS["curves"](tracker, history, prefix=f"{model_type}_L{max_len}_")

            # histograma longitudes (sobre un subconjunto para no tardar)
            if tracker.enabled and ("length_hist" in plotter_names):
                lengths = [min(len(tok(t)) + 1, max_len) for t in X_va[:5000]]
                PLOTTERS["length_hist"](tracker, lengths, prefix=f"{model_type}_L{max_len}_")

        # test (con gather) + confusion matrix
        if do_test:
            model.eval()
            test_preds_local, test_gts_local = [], []
            with torch.no_grad():
                for batch in test_loader:
                    ids  = batch["input_ids"].cuda(local_rank, non_blocking=True)
                    attn = batch["attention_mask"].cuda(local_rank, non_blocking=True)
                    y    = batch["labels"].cuda(local_rank, non_blocking=True)
                    if use_amp:
                        with torch.amp.autocast('cuda', enabled=True):
                            _, logits = model(ids, attn, None)
                    else:
                        _, logits = model(ids, attn, None)
                    test_preds_local.append(logits.argmax(-1))
                    test_gts_local.append(y)

            test_preds_local = torch.cat(test_preds_local) if test_preds_local else torch.empty(0, dtype=torch.long, device=local_rank)
            test_gts_local   = torch.cat(test_gts_local)   if test_gts_local   else torch.empty(0, dtype=torch.long, device=local_rank)

            preds_all = all_gather_1d(test_preds_local, world_size, -1)
            gts_all   = all_gather_1d(test_gts_local,   world_size, -1)

            if is_main:
                p = torch.cat(preds_all).cpu().numpy()
                g = torch.cat(gts_all).cpu().numpy()
                m = (g != -1)
                p, g = p[m], g[m]

                test_acc = accuracy_score(g, p) if p.size else 0.0
                test_f1  = f1_score(g, p, average="macro") if p.size else 0.0
                tracker.log_metrics({"test_accuracy": test_acc, "test_macro_f1": test_f1}, step=epochs)

                metrics = {"test_accuracy": test_acc, "test_macro_f1": test_f1}

                tracker.log_metrics(metrics, step=epochs)

                if tracker.enabled and ("confusion" in plotter_names):
                    cm = confusion_matrix(g, p, labels=list(range(num_labels)))
                    PLOTTERS["confusion"](tracker, cm, class_names=label_names, prefix=f"{model_type}_L{max_len}_")

        # run summary (muy útil para debug / auditoría)
        if is_main and tracker.enabled:
            tracker.log_dict({
                "config": cfg,
                "history": history,
                "best_val_f1": float(best_f1),
            }, artifact_file=f"{tracker.artifact_dir}/run_summary.json")

        if tracker:
            tracker.end()

    except Exception as e:
        print(f"[EXCEPTION] {repr(e)}", flush=True)
        traceback.print_exc()
        raise
    finally:
        if dist_inited:
            ddp_cleanup()

In [0]:
# COMMAND ----------
# quick sanity (no DDP): solo para validar shapes
(trX, trY), (vaX, vaY), _, num_labels = load_20news("/tmp/sklearn_20news")
stoi, itos = build_vocab_deterministic(trX)
ds = TextDS(trX[:64], trY[:64], stoi, max_len=128)
dl = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate)

batch = next(iter(dl))
print(batch["input_ids"].shape, batch["attention_mask"].shape, batch["labels"].shape)

m = build_model("ffn", len(itos), num_labels, 128).cuda()
loss, logits = m(batch["input_ids"].cuda(), batch["attention_mask"].cuda(), batch["labels"].cuda())
print(loss.item(), logits.shape)

torch.Size([8, 128]) torch.Size([8, 128]) torch.Size([8])
2.9751739501953125 torch.Size([8, 20])


In [0]:
with torch.no_grad():
    loss1, logits1 = m(batch["input_ids"].cuda(), batch["attention_mask"].cuda(), batch["labels"].cuda())
    # "sin máscara" (equivale a mean pooling sin ignorar PAD en tu FFN)
    loss2, logits2 = m(batch["input_ids"].cuda(), None, batch["labels"].cuda())
print("loss masked:", float(loss1), "loss no_mask:", float(loss2))

loss masked: 2.980156183242798 loss no_mask: 2.9761881828308105


In [0]:
# COMMAND ----------
def main_worker(cfg: dict):
    train_ddp_one_cfg(cfg)

# def run_cfg_distributed(cfg: dict, num_processes: int, local_mode: bool):
#     print(f"[launcher] local_mode={local_mode} num_processes={num_processes} cfg={cfg}", flush=True)
#     return TorchDistributor(num_processes=num_processes, local_mode=local_mode, use_gpu=True).run(main_worker, cfg)

def run_cfg_distributed(cfg: dict, num_processes: int, local_mode: bool):
    visible = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print(f"[launcher] CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')} visible_gpus={visible}", flush=True)

    if local_mode and cfg.get("use_amp", True) and torch.cuda.is_available():
        if num_processes > visible:
            raise ValueError(
                f"num_processes={num_processes} pero solo hay {visible} GPU(s) visibles en este Job run. "
                "Ajusta num_processes o el compute del Job."
            )

    print(f"[launcher] local_mode={local_mode} num_processes={num_processes} cfg={cfg}", flush=True)
    return TorchDistributor(num_processes=num_processes, local_mode=local_mode, use_gpu=True).run(main_worker, cfg)

# def run_cfg_single_gpu(cfg):
#     import os

#     # 1) fijar GPU física asignada por el grid
#     gpu_id = str(cfg.get("gpu_ids", "0"))
#     os.environ["CUDA_VISIBLE_DEVICES"] = gpu_id

#     # 2) IMPORTANTE: importar torch DESPUÉS de fijar CUDA_VISIBLE_DEVICES
#     import torch

#     # 3) en este proceso solo existirá cuda:0 (mapeado a la GPU física gpu_id)
#     assert torch.cuda.is_available(), "CUDA no disponible"
#     torch.cuda.set_device(0)

#     # 4) llamar a tu entrenamiento SIN TorchDistributor
#     return train_one_cfg(cfg)   # 👈 ver siguiente sección (train_one_cfg)

In [0]:
# COMMAND ----------
from pyspark.ml.torch.distributor import TorchDistributor

def to_int(x: str) -> int:
    # acepta "13", "13.0", "13\n", etc.
    return int(float(x))

def to_float(x: str) -> float:
    return float(x)

cfg = {
    "model_type": dbutils.widgets.get("model_type"),
    "seed": to_int(dbutils.widgets.get("seed")),
    "lr": to_float(dbutils.widgets.get("lr")),
    "max_len": to_int(dbutils.widgets.get("max_len")),
    "base_batch_size": to_int(dbutils.widgets.get("base_batch_size")),
    "epochs": to_int(dbutils.widgets.get("epochs")),
    "experiment_path": dbutils.widgets.get("experiment_path"),
    "data_home": dbutils.widgets.get("data_home"),
    "enable_mlflow": dbutils.widgets.get("enable_mlflow").lower() == "true",
    "use_amp": False,
    "dataset": dbutils.widgets.get("dataset")
}
num_processes = to_int(dbutils.widgets.get("num_processes"))
local_mode = dbutils.widgets.get("local_mode").lower() == "true"

cfg["extra_params"] = {
  "dropout": 0.1,
  "arch": ARCH,   # anidado OK (se aplana)
  "note": "prueba con logging extendido"
}
cfg["tags"] = {"project": "20news", "owner": "david.grana"}
cfg["plotters"] = ["curves", "confusion"]          # solo estas
cfg["artifact_dir"] = "viz"                        # cambia carpeta artifacts/viz/...
# cfg["log_every_epochs"] = 1                        # log por epoch (recomendado)
cfg["do_test"] = True                              # activa test + confusion

num_processes = to_int(dbutils.widgets.get("num_processes"))
local_mode = dbutils.widgets.get("local_mode").lower() == "true"

if local_mode and "CUDA_VISIBLE_DEVICES" in os.environ:
    print("Unsetting CUDA_VISIBLE_DEVICES for TorchDistributor local_mode=True:",
          os.environ["CUDA_VISIBLE_DEVICES"], flush=True)
    del os.environ["CUDA_VISIBLE_DEVICES"]

run_cfg_distributed(cfg, num_processes=num_processes, local_mode=local_mode)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

[launcher] CUDA_VISIBLE_DEVICES=None visible_gpus=4
[launcher] local_mode=True num_processes=4 cfg={'model_type': 'ffn', 'seed': 13, 'lr': 0.002, 'max_len': 512, 'base_batch_size': 64, 'epochs': 20, 'experiment_path': '/Shared/project_test', 'data_home': '/tmp/sklearn_20news', 'enable_mlflow': True, 'use_amp': True, 'dataset': '20news', 'extra_params': {'dropout': 0.2, 'arch': {'emb_dim': 128, 'hidden': 256}, 'note': 'prueba con logging extendido'}, 'tags': {'project': '20news', 'owner': 'david.grana'}, 'plotters': ['curves', 'confusion'], 'artifact_dir': 'viz', 'do_test': True}


Started local training with 4 processes
Finished local training with 4 processes


Sun Feb 22 16:12:41 2026 Connection to spark from PID  117456
Sun Feb 22 16:12:41 2026 Initialized gateway on port 46061
Sun Feb 22 16:12:41 2026 Connected to spark.
2026/02/22 16:12:41 INFO mlflow.tracking.fluent: Experiment with name '/Shared/project_test' does not exist. Creating a new experiment.
Uploading artifacts: 100%|██████████| 1/1 [00:00<00:00,  3.99it/s]
[rank0] ep=1 train_loss=2.8515 val_loss=2.5990 val_acc=0.2438 val_f1=0.2085 val_rec=0.2369
Uploading artifacts: 100%|██████████| 1/1 [00:00<00:00,  5.09it/s]
[rank0] ep=2 train_loss=2.2477 val_loss=2.0412 val_acc=0.3613 val_f1=0.3077 val_rec=0.3490
Uploading artifacts: 100%|██████████| 1/1 [00:00<00:00,  4.64it/s]
[rank0] ep=3 train_loss=1.7323 val_loss=1.6944 val_acc=0.4788 val_f1=0.4352 val_rec=0.4616
Uploading artifacts: 100%|██████████| 1/1 [00:00<00:00,  5.08it/s]
[rank0] ep=4 train_loss=1.3289 val_loss=1.4771 val_acc=0.5477 val_f1=0.5263 val_rec=0.5334
Uploading artifacts: 100%|██████████| 1/1 [00:00<00:00,  5.25it/s]